# 07 — Speed & Distance Estimation

We now have everything we need:
- Real-world foot positions in metres (after perspective transform + camera compensation)
- A known frame rate (24 fps)

From these we can compute speed in km/h and total distance covered.

**Key formula:**
```
speed (m/s)  = Euclidean_distance(pos_start, pos_end) / elapsed_seconds
speed (km/h) = speed (m/s) × 3.6
```

**Connection to the pipeline:** `football_analysis/speed_distance_estimator/speed_distance_estimator.py`

In [ ]:
import sys
sys.path.append('..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from football_analysis.trackers.tracker import Tracker
from football_analysis.camera_movement_estimator.camera_movement_estimator import CameraMovementEstimator
from football_analysis.view_transformer.view_transformer import ViewTransformer
from football_analysis.speed_distance_estimator.speed_distance_estimator import SpeedAndDistanceEstimator

In [ ]:
VIDEO_PATH     = '../input_videos/08fd33_4.mp4'
MODEL_PATH     = '../models/best.pt'
TRACK_STUB     = '../stubs/track_stubs.pkl'
CAM_STUB       = '../stubs/camera_movement_stub.pkl'

cap = cv2.VideoCapture(VIDEO_PATH)
ret, frame0 = cap.read()
cap.release()

# Load everything from stubs
tracker = Tracker(MODEL_PATH)
tracks  = tracker.get_object_tracks([], read_from_stub=True, stub_path=TRACK_STUB)
tracker.add_position_to_tracks(tracks)

cam_est = CameraMovementEstimator(frame0)
camera_movement = cam_est.get_camera_movement([], read_from_stub=True, stub_path=CAM_STUB)
cam_est.add_adjust_positions_to_tracks(tracks, camera_movement)

ViewTransformer().add_transformed_position_to_tracks(tracks)

print(f'Tracks loaded. Frames: {len(tracks["players"])}')

## The core calculation

We compute speed over a **window of frames** (default: 5) rather than frame-to-frame.  
Frame-to-frame speed is extremely noisy (tracker jitter, sub-pixel errors).  
A 5-frame window at 24 fps = ~0.2 seconds — smooth enough to be meaningful.

In [ ]:
FRAME_RATE   = 24
FRAME_WINDOW = 5

# Manually compute speed for one player to understand the maths
# Find a player that appears frequently
from collections import Counter
pid_counts = Counter()
for frame_tracks in tracks['players']:
    pid_counts.update(frame_tracks.keys())

target_pid = pid_counts.most_common(1)[0][0]
print(f'Analysing player ID {target_pid} (appears in {pid_counts[target_pid]} frames)')

In [ ]:
# Collect real-world positions for this player across all frames
positions = {}   # frame_num → (x_m, y_m)

for frame_num, frame_tracks in enumerate(tracks['players']):
    if target_pid in frame_tracks:
        pos = frame_tracks[target_pid].get('position_transformed')
        if pos is not None:
            positions[frame_num] = pos

frame_nums = sorted(positions.keys())
print(f'Valid positions: {len(frame_nums)} frames')
print(f'First position: frame {frame_nums[0]} → {positions[frame_nums[0]]}')

In [ ]:
# Compute speed over a sliding window
speeds    = []
distances = []
cumulative_dist = 0.0

for i in range(len(frame_nums)):
    start_frame = frame_nums[i]
    # Find end frame that is FRAME_WINDOW ahead
    end_frame = start_frame + FRAME_WINDOW
    if end_frame not in positions:
        continue

    x0, y0 = positions[start_frame]
    x1, y1 = positions[end_frame]

    dist_m   = np.sqrt((x1 - x0)**2 + (y1 - y0)**2)   # Euclidean distance in metres
    time_s   = FRAME_WINDOW / FRAME_RATE                 # elapsed seconds
    speed_ms = dist_m / time_s                           # metres per second
    speed_kh = speed_ms * 3.6                            # km/h

    cumulative_dist += dist_m

    speeds.append({'frame': start_frame, 'speed_kmh': speed_kh})
    distances.append({'frame': start_frame, 'cumulative_m': cumulative_dist})

speed_df = pd.DataFrame(speeds)
dist_df  = pd.DataFrame(distances)

print(f'\nPlayer {target_pid} stats:')
print(f'  Max speed    : {speed_df["speed_kmh"].max():.1f} km/h')
print(f'  Avg speed    : {speed_df["speed_kmh"].mean():.1f} km/h')
print(f'  Total distance: {cumulative_dist:.1f} m')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(speed_df['frame'], speed_df['speed_kmh'], color='steelblue')
axes[0].axhline(speed_df['speed_kmh'].mean(), color='tomato', linestyle='--', label='mean')
axes[0].set_ylabel('Speed (km/h)')
axes[0].set_title(f'Player {target_pid} — speed over time')
axes[0].legend()

axes[1].plot(dist_df['frame'], dist_df['cumulative_m'], color='green')
axes[1].set_ylabel('Distance (m)')
axes[1].set_xlabel('Frame')
axes[1].set_title(f'Player {target_pid} — cumulative distance')

plt.tight_layout()
plt.show()

## Running the pipeline's own estimator

In [ ]:
SpeedAndDistanceEstimator().add_speed_and_distance_to_tracks(tracks)

# Collect stats for all players
player_stats = {}
for frame_tracks in tracks['players']:
    for pid, info in frame_tracks.items():
        if pid not in player_stats:
            player_stats[pid] = {'speeds': [], 'max_dist': 0.0}
        if 'speed' in info:
            player_stats[pid]['speeds'].append(info['speed'])
        player_stats[pid]['max_dist'] = max(player_stats[pid]['max_dist'], info.get('distance', 0.0))

rows = []
for pid, data in player_stats.items():
    if data['speeds']:
        rows.append({
            'player_id'    : pid,
            'avg_speed_kmh': round(np.mean(data['speeds']), 1),
            'max_speed_kmh': round(max(data['speeds']), 1),
            'distance_m'   : round(data['max_dist'], 1),
        })

df = pd.DataFrame(rows).sort_values('max_speed_kmh', ascending=False)
print(df.to_string(index=False))

## Sanity check: are these realistic?

Elite players sprint at 30-35 km/h. Average match speed ~10-12 km/h.  
In a 30-second clip we'd expect distances of 50-100 m for active players.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(df['player_id'].astype(str), df['max_speed_kmh'], color='steelblue')
axes[0].axvline(35, color='red', linestyle='--', label='Elite sprint (~35 km/h)')
axes[0].set_xlabel('Max speed (km/h)')
axes[0].set_title('Max speed per player')
axes[0].legend()

axes[1].barh(df['player_id'].astype(str), df['distance_m'], color='green')
axes[1].set_xlabel('Distance (m)')
axes[1].set_title('Distance covered per player')

plt.tight_layout()
plt.show()

## Pipeline summary — all 7 components together

| Notebook | Component | Input → Output |
|---|---|---|
| 01 | Video I/O | MP4 file → list of BGR frames |
| 02 | Detection | Frames → bounding boxes + class labels |
| 03 | Tracking | Detections → stable player IDs across frames |
| 04 | Team assignment | Player crops → team 1 or team 2 label |
| 05 | Camera movement | Optical flow on sidelines → (dx, dy) per frame |
| 06 | Perspective transform | Pixel positions → real-world metres |
| 07 | Speed & distance | Positions × time → km/h, total metres |

Everything is assembled in `main.py` and exposed as a Streamlit app in `app.py`.

## Key takeaways

| Concept | Detail |
|---|---|
| Window smoothing | 5-frame window reduces tracker jitter vs. frame-to-frame |
| Unit conversion | `m/s × 3.6 = km/h` |
| Accumulation | Total distance sums window-level displacements (not positions directly) |
| Missing positions | Players outside the homography quad have no `position_transformed` — speed not computed |

**You now understand every component in the pipeline.** 🎉